In [1]:
# Extract hosp-module histories that finish before each patient's first ICU stay.
# Visit metadata includes age, admission type, and day of year.

# GT-BEHRT Data Preprocessing

This notebook converts the MIMIC-IV-Data-Pipeline CSV outputs into the pickled `data` object expected by `GT-BEHRT_Notebook.ipynb`.

The output structure is:

`dataset[patient_index][visit_index] -> torch_geometric.data.Data`

Each patient is padded/truncated to 50 visit graphs.

By default, this notebook processes a sample of patients. Change `SAMPLE_N_PATIENTS` in the first code cell, or set it to `None` for the full cohort.


In [2]:
from pathlib import Path
import itertools
import pickle

import pandas as pd
import torch
from torch_geometric.data import Data

PIPELINE_DIR = Path('/home/mingzhul/projects/Simultaneous-EHR/repos/MIMIC-IV-Data-Pipeline/data')
RAW_MIMIC_HOSP_DIR = Path('/home/mingzhul/projects/Simultaneous-EHR/repos/MIMIC-IV-Data-Pipeline/mimiciv/3.1/hosp')
OUT_DIR = Path('/home/mingzhul/projects/Simultaneous-EHR/repos/GT-BEHRT')

ICU_COHORT_PATH = PIPELINE_DIR / 'cohort/cohort_icu_mortality_0_.csv.gz'
DIAG_PATH = PIPELINE_DIR / 'features/preproc_diag.csv'
MED_PATH = PIPELINE_DIR / 'features/preproc_med.csv'
PROC_PATH = PIPELINE_DIR / 'features/preproc_proc.csv'
ADMISSIONS_PATH = RAW_MIMIC_HOSP_DIR / 'admissions.csv.gz'
PATIENTS_PATH = RAW_MIMIC_HOSP_DIR / 'patients.csv.gz'

MAX_SEQ_LEN = 50
VST_ID = 0
# Count real admissions only. The synthetic <VST> graph node does not count.
MIN_REAL_VISITS_PER_PATIENT = 1

# Set to None to process the full cohort. Keep this small while debugging.
SAMPLE_N_PATIENTS = None
SAMPLE_RANDOM_SEED = 42
CSV_CHUNK_ROWS = 1_000_000

OUT_DIR = OUT_DIR / f'sample_{SAMPLE_N_PATIENTS}/'
OUT_DIR.mkdir(parents=True, exist_ok=True)


first_icu_all = pd.read_csv(ICU_COHORT_PATH)
first_icu_all['intime'] = pd.to_datetime(first_icu_all['intime'])
first_icu_all['outtime'] = pd.to_datetime(first_icu_all['outtime'])
first_icu_all = (
    first_icu_all.sort_values(['subject_id', 'intime', 'stay_id'])
    .groupby('subject_id', as_index=False)
    .first()
    .rename(
        columns={
            'stay_id': 'icu_stay_id',
            'hadm_id': 'icu_hadm_id',
            'intime': 'icu_intime',
            'outtime': 'icu_outtime',
            'Age': 'icu_age',
        }
    )
)

if SAMPLE_N_PATIENTS is None:
    first_icu = first_icu_all.copy()
else:
    unique_subjects = first_icu_all['subject_id'].drop_duplicates()
    n = min(SAMPLE_N_PATIENTS, len(unique_subjects))
    sampled_subjects = set(unique_subjects.sample(n=n, random_state=SAMPLE_RANDOM_SEED))
    first_icu = first_icu_all[first_icu_all['subject_id'].isin(sampled_subjects)].copy()

patients = pd.read_csv(PATIENTS_PATH, usecols=['subject_id', 'anchor_year', 'anchor_age'])
patients['yob'] = patients['anchor_year'] - patients['anchor_age']
first_icu = first_icu.merge(patients[['subject_id', 'yob']], on='subject_id', how='left')

admissions_all = pd.read_csv(
    ADMISSIONS_PATH,
    usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type'],
)
admissions_all['admittime'] = pd.to_datetime(admissions_all['admittime'])
admissions_all['dischtime'] = pd.to_datetime(admissions_all['dischtime'])

cohort = admissions_all.merge(
    first_icu[
        ['subject_id', 'icu_stay_id', 'icu_hadm_id', 'icu_intime', 'icu_outtime', 'icu_age', 'label', 'yob']
    ],
    on='subject_id',
    how='inner',
)
cohort = cohort[cohort['dischtime'] < cohort['icu_intime']].copy()
cohort['visit_age'] = cohort['admittime'].dt.year - cohort['yob']
cohort['visit_age'] = cohort['visit_age'].fillna(
    cohort['icu_age'] - (cohort['icu_intime'].dt.year - cohort['admittime'].dt.year)
)

adm_type_values = sorted(cohort['admission_type'].dropna().unique())
adm_type_vocab = {value: idx + 1 for idx, value in enumerate(adm_type_values)}
assert len(adm_type_vocab) <= 10, len(adm_type_vocab)
cohort['adm_type_id'] = cohort['admission_type'].map(adm_type_vocab).fillna(0).astype(int)

selected_hadm_ids = set(cohort['hadm_id'].unique())
selected_subject_ids = set(first_icu['subject_id'].unique())

def read_feature_subset(path, usecols, label):
    chunks = []
    total_rows = 0
    kept_rows = 0

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CSV_CHUNK_ROWS):
        total_rows += len(chunk)
        chunk = chunk[chunk['hadm_id'].isin(selected_hadm_ids)]
        kept_rows += len(chunk)
        if not chunk.empty:
            chunks.append(chunk.copy())

    if chunks:
        out = pd.concat(chunks, ignore_index=True)
    else:
        out = pd.DataFrame(columns=usecols)

    print(f'{label}: kept {kept_rows:,}/{total_rows:,} rows')
    return out

diagnosis = read_feature_subset(DIAG_PATH, ['subject_id', 'hadm_id', 'new_icd_code'], 'diagnosis')
medication = read_feature_subset(MED_PATH, ['subject_id', 'hadm_id', 'drug_name'], 'medication')
procedure = read_feature_subset(PROC_PATH, ['subject_id', 'hadm_id', 'icd_code'], 'procedure')

print('first ICU patients:', len(selected_subject_ids))
print('pre-ICU admissions:', len(selected_hadm_ids))
print('cohort:', cohort.shape)
print('diagnosis:', diagnosis.shape)
print('medication:', medication.shape)
print('procedure:', procedure.shape)
print('admission types:', adm_type_vocab)


/home/mingzhul/projects/Simultaneous-EHR/repos/GT-BEHRT/.conda/ml_gt-behrt/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


diagnosis: kept 700,080/6,055,655 rows
medication: kept 1,041,090/11,975,319 rows
procedure: kept 22,182/390,446 rows
first ICU patients: 65355
pre-ICU admissions: 62727
cohort: (62727, 14)
diagnosis: (700080, 3)
medication: (1041090, 3)
procedure: (22182, 3)
admission types: {'AMBULATORY OBSERVATION': 1, 'DIRECT EMER.': 2, 'DIRECT OBSERVATION': 3, 'ELECTIVE': 4, 'EU OBSERVATION': 5, 'EW EMER.': 6, 'OBSERVATION ADMIT': 7, 'SURGICAL SAME DAY ADMISSION': 8, 'URGENT': 9}


In [3]:
cohort_all

NameError: name 'cohort_all' is not defined

## Build Typed Visit Events

GT-BEHRT needs to know whether each code is diagnosis, medication, or procedure when it builds edge types. To avoid a separate `id_to_type` dictionary, each visit stores typed tuples: `(code_id, code_type)`.

In [4]:
def make_events(df, code_col, code_type):
    events = df[['subject_id', 'hadm_id', code_col]].copy()
    events = events.dropna(subset=[code_col])
    events['code'] = code_type + ':' + events[code_col].astype(str).str.strip().str.lower()
    events['code_type'] = code_type
    events = events[['subject_id', 'hadm_id', 'code', 'code_type']]
    return events.drop_duplicates()

diag_events = make_events(diagnosis, 'new_icd_code', 'diag')
med_events = make_events(medication, 'drug_name', 'med')
proc_events = make_events(procedure, 'icd_code', 'proc')

events = pd.concat([diag_events, med_events, proc_events], ignore_index=True)
events = events.drop_duplicates(['subject_id', 'hadm_id', 'code', 'code_type'])

code_vocab = {code: idx for idx, code in enumerate(sorted(events['code'].unique()))}
events['code_id'] = events['code'].map(code_vocab).astype(int)

typed_codes_by_hadm = (
    events.groupby(['subject_id', 'hadm_id'])[['code_id', 'code_type']]
    .apply(lambda x: sorted(set((int(r.code_id), r.code_type) for r in x.itertuples())))
    .rename('typed_codes')
    .reset_index()
)

print('unique codes:', len(code_vocab))
print('GT-BEHRT vocab_size should be:', len(code_vocab) + 1)
print('admissions with at least one code:', len(typed_codes_by_hadm))

unique codes: 5857
GT-BEHRT vocab_size should be: 5858
admissions with at least one code: 62677


## Build Patient Records

This intermediate `combined` object keeps records human-readable before conversion to PyG graphs:

`[subject_id, [label, los_days], visits]`

Each visit is:

`[[delta_days], [(code_id, code_type), ...], metadata]`

In [5]:
def clamp_int(value, low, high, default=0):
    try:
        value = int(value)
    except (TypeError, ValueError):
        value = default
    return max(low, min(high, value))

admissions = cohort.merge(typed_codes_by_hadm, on=['subject_id', 'hadm_id'], how='left')
admissions = admissions[admissions['typed_codes'].notna()].copy()
admissions = admissions.sort_values(['subject_id', 'admittime'])

combined = []

for subject_id, group in admissions.groupby('subject_id', sort=False):
    group = group.sort_values('admittime')
    first_admit = group['admittime'].iloc[0]
    patient_label = int(group['label'].iloc[0])
    visits = []

    for row in group.itertuples(index=False):
        typed_codes = list(row.typed_codes)
        if not typed_codes:
            continue

        delta_days = max(0, int((row.admittime - first_admit).days))
        los_days = max(0, int((row.dischtime - row.admittime).total_seconds() // 86400))
        metadata = {
            'age': clamp_int(row.visit_age, 0, 102),
            'day_of_year': clamp_int(row.admittime.dayofyear, 0, 366),
            'los': clamp_int(los_days, 0, 1191),
            'adm_type': clamp_int(row.adm_type_id, 0, 10),
        }
        visits.append([[delta_days], typed_codes, metadata])

    # `visits` counts real admissions with codes, not the synthetic <VST> node inside each graph.
    if len(visits) >= MIN_REAL_VISITS_PER_PATIENT:
        last_los = visits[-1][2]['los']
        combined.append([int(subject_id), [patient_label, last_los], visits[:MAX_SEQ_LEN]])

print(f'patients with >= {MIN_REAL_VISITS_PER_PATIENT} visits:', len(combined))
print('first patient visits:', len(combined[0][2]) if combined else 0)
print('first visit example:', combined[0][2][0] if combined else None)

patients with >= 1 visits: 21493
first patient visits: 2
first visit example: [[0], [(46, 'diag'), (331, 'diag'), (340, 'diag'), (590, 'diag'), (667, 'diag'), (672, 'diag'), (1014, 'diag'), (1382, 'diag'), (1416, 'med'), (1715, 'med'), (1802, 'med'), (1825, 'med'), (1883, 'med'), (2050, 'med'), (2194, 'med'), (2270, 'med')], {'age': 52, 'day_of_year': 127, 'los': 0, 'adm_type': 9}]


## Convert Visits To GT-BEHRT Graphs

The GT-BEHRT model reserves node id `0` for the visit/readout node. Every medical code id is shifted by `+1` during graph construction.

In [6]:
EDGE_TYPES = {
    ('diag', 'diag'): 0,
    ('med', 'med'): 1,
    ('proc', 'proc'): 2,
    ('diag', 'med'): 3,
    ('diag', 'proc'): 4,
    ('med', 'proc'): 5,
    ('vst', 'code'): 6,
}

def normalize_pair(a, b):
    if a == 'vst' or b == 'vst':
        return ('vst', 'code')
    return tuple(sorted((a, b)))

def make_visit_graph(typed_codes, delta, label, age=0, day_of_year=0, adm_type=0, los=0):
    x = [VST_ID] + [int(code_id) + 1 for code_id, _ in typed_codes]
    node_types = ['vst'] + [code_type for _, code_type in typed_codes]
    n_nodes = len(x)

    edges = []
    edge_attrs = []
    for src, dst in itertools.permutations(range(n_nodes), 2):
        pair = normalize_pair(node_types[src], node_types[dst])
        edges.append([src, dst])
        edge_attrs.append(EDGE_TYPES[pair])

    if not edges:
        edges = [[0, 0]]
        edge_attrs = [EDGE_TYPES[('vst', 'code')]]

    return Data(
        x=torch.tensor(x, dtype=torch.long),
        edge_index=torch.tensor(edges, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attrs, dtype=torch.long),
        age=torch.tensor([clamp_int(age, 0, 102)], dtype=torch.long),
        time=torch.tensor([clamp_int(day_of_year, 0, 366)], dtype=torch.long),
        delta=torch.tensor([clamp_int(delta, 0, 143)], dtype=torch.long),
        adm_type=torch.tensor([clamp_int(adm_type, 0, 10)], dtype=torch.long),
        los=torch.tensor([clamp_int(los, 0, 1191)], dtype=torch.long),
        mask_v=torch.tensor([1], dtype=torch.long),
        label=torch.tensor([int(label)], dtype=torch.float),
        mask=torch.tensor([1], dtype=torch.long),
    )

def make_pad_graph(label):
    return Data(
        x=torch.tensor([VST_ID], dtype=torch.long),
        edge_index=torch.tensor([[0], [0]], dtype=torch.long),
        edge_attr=torch.tensor([EDGE_TYPES[('vst', 'code')]], dtype=torch.long),
        age=torch.tensor([0], dtype=torch.long),
        time=torch.tensor([0], dtype=torch.long),
        delta=torch.tensor([0], dtype=torch.long),
        adm_type=torch.tensor([0], dtype=torch.long),
        los=torch.tensor([0], dtype=torch.long),
        mask_v=torch.tensor([0], dtype=torch.long),
        label=torch.tensor([int(label)], dtype=torch.float),
        mask=torch.tensor([1], dtype=torch.long),
    )

def convert_patient(patient):
    subject_id, label_obj, visits = patient
    label = int(label_obj[0])
    graphs = []

    for visit in visits[:MAX_SEQ_LEN]:
        delta = int(visit[0][0])
        typed_codes = visit[1]
        metadata = visit[2] if len(visit) > 2 else {}
        graphs.append(
            make_visit_graph(
                typed_codes=typed_codes,
                delta=delta,
                label=label,
                age=metadata.get('age', 0),
                day_of_year=metadata.get('day_of_year', 0),
                adm_type=metadata.get('adm_type', 0),
                los=metadata.get('los', 0),
            )
        )

    while len(graphs) < MAX_SEQ_LEN:
        graphs.append(make_pad_graph(label))

    return graphs

## Save GT-BEHRT Dataset

The training notebook's cell 5 should load `OUT_DIR / 'data'`.

In [7]:
# len(combined[4][2])
combined

[[10000032,
  [0, 1],
  [[[0],
    [(46, 'diag'),
     (331, 'diag'),
     (340, 'diag'),
     (590, 'diag'),
     (667, 'diag'),
     (672, 'diag'),
     (1014, 'diag'),
     (1382, 'diag'),
     (1416, 'med'),
     (1715, 'med'),
     (1802, 'med'),
     (1825, 'med'),
     (1883, 'med'),
     (2050, 'med'),
     (2194, 'med'),
     (2270, 'med')],
    {'age': 52, 'day_of_year': 127, 'los': 0, 'adm_type': 9}],
   [[50],
    [(46, 'diag'),
     (231, 'diag'),
     (304, 'diag'),
     (315, 'diag'),
     (590, 'diag'),
     (667, 'diag'),
     (1014, 'diag'),
     (1328, 'diag'),
     (1416, 'med'),
     (1715, 'med'),
     (1802, 'med'),
     (1825, 'med'),
     (1904, 'med'),
     (2194, 'med'),
     (2207, 'med'),
     (2278, 'med'),
     (2316, 'med')],
    {'age': 52, 'day_of_year': 178, 'los': 1, 'adm_type': 6}]]],
 [10000690,
  [0, 7],
  [[[0],
    [(478, 'diag'),
     (489, 'diag'),
     (508, 'diag'),
     (523, 'diag'),
     (1023, 'diag'),
     (1024, 'diag'),
     (1057, 'd

In [8]:
import gc

# Save the lighter, human-readable intermediate before converting it in place.
# The in-place conversion avoids holding both `combined` and `gt_behrt_dataset`
# as two giant lists at the same time.
with open(OUT_DIR / 'pipeline_output.combined.train', 'wb') as f:
    pickle.dump(combined, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(OUT_DIR / 'code_vocab.pkl', 'wb') as f:
    pickle.dump(code_vocab, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(OUT_DIR / 'adm_type_vocab.pkl', 'wb') as f:
    pickle.dump(adm_type_vocab, f, protocol=pickle.HIGHEST_PROTOCOL)

first_icu[['subject_id', 'icu_stay_id', 'icu_hadm_id', 'icu_intime', 'label']].to_csv(
    OUT_DIR / 'icu_index.csv',
    index=False,
)

patient_count = len(combined)
vocab_size = len(code_vocab) + 1

for idx, patient in enumerate(combined):
    combined[idx] = convert_patient(patient)
    if idx and idx % 1000 == 0:
        gc.collect()
        print(f'converted {idx:,}/{patient_count:,} patients')

# Keep only a tiny sample for validation after saving, then release the full list.
sample_graph = combined[0][0] if combined else None
sample_patient_len = len(combined[0]) if combined else 0

with open(OUT_DIR / 'data', 'wb') as f:
    pickle.dump(combined, f, protocol=pickle.HIGHEST_PROTOCOL)

print('saved:', OUT_DIR / 'data')
print('patients:', patient_count)
print('visits per patient:', sample_patient_len)
print('vocab_size for GT-BEHRT_Notebook:', vocab_size)
print('saved admission type vocab:', OUT_DIR / 'adm_type_vocab.pkl')
print('saved ICU anchors:', OUT_DIR / 'icu_index.csv')

# # # Drop the largest objects immediately. Re-load OUT_DIR / 'data' later if needed.
# del combined
# gc.collect()


converted 1,000/21,493 patients
converted 2,000/21,493 patients
converted 3,000/21,493 patients
converted 4,000/21,493 patients
converted 5,000/21,493 patients
converted 6,000/21,493 patients
converted 7,000/21,493 patients
converted 8,000/21,493 patients
converted 9,000/21,493 patients
converted 10,000/21,493 patients
converted 11,000/21,493 patients
converted 12,000/21,493 patients
converted 13,000/21,493 patients
converted 14,000/21,493 patients
converted 15,000/21,493 patients
converted 16,000/21,493 patients
converted 17,000/21,493 patients
converted 18,000/21,493 patients
converted 19,000/21,493 patients
converted 20,000/21,493 patients
converted 21,000/21,493 patients
saved: /home/mingzhul/projects/Simultaneous-EHR/repos/GT-BEHRT/sample_None/data
patients: 21493
visits per patient: 50
vocab_size for GT-BEHRT_Notebook: 5858
saved admission type vocab: /home/mingzhul/projects/Simultaneous-EHR/repos/GT-BEHRT/sample_None/adm_type_vocab.pkl
saved ICU anchors: /home/mingzhul/projects/

In [9]:
combined[245]

[Data(x=[25], edge_index=[2, 600], edge_attr=[600], age=[1], time=[1], delta=[1], adm_type=[1], los=[1], mask_v=[1], label=[1], mask=[1]),
 Data(x=[9], edge_index=[2, 72], edge_attr=[72], age=[1], time=[1], delta=[1], adm_type=[1], los=[1], mask_v=[1], label=[1], mask=[1]),
 Data(x=[40], edge_index=[2, 1560], edge_attr=[1560], age=[1], time=[1], delta=[1], adm_type=[1], los=[1], mask_v=[1], label=[1], mask=[1]),
 Data(x=[1], edge_index=[2, 1], edge_attr=[1], age=[1], time=[1], delta=[1], adm_type=[1], los=[1], mask_v=[1], label=[1], mask=[1]),
 Data(x=[1], edge_index=[2, 1], edge_attr=[1], age=[1], time=[1], delta=[1], adm_type=[1], los=[1], mask_v=[1], label=[1], mask=[1]),
 Data(x=[1], edge_index=[2, 1], edge_attr=[1], age=[1], time=[1], delta=[1], adm_type=[1], los=[1], mask_v=[1], label=[1], mask=[1]),
 Data(x=[1], edge_index=[2, 1], edge_attr=[1], age=[1], time=[1], delta=[1], adm_type=[1], los=[1], mask_v=[1], label=[1], mask=[1]),
 Data(x=[1], edge_index=[2, 1], edge_attr=[1], a

In [10]:
import pickle
import pandas as pd

path = OUT_DIR / 'pipeline_output.combined.train'

with path.open('rb') as f:
    combined = pickle.load(f)

los_values = []

for patient in combined:
    visits = patient[2]
    for visit in visits:
        metadata = visit[2]
        los_values.append(metadata['los'])

los = pd.Series(los_values, name='los_days')

print(los.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print('num visits:', len(los))
print('num zero LOS:', int((los == 0).sum()))
print('pct zero LOS:', float((los == 0).mean()))
print('max LOS:', int(los.max()))

count    62635.000000
mean         3.608781
std          5.671187
min          0.000000
1%           0.000000
5%           0.000000
25%          1.000000
50%          2.000000
75%          4.000000
95%         12.000000
99%         27.000000
max        170.000000
Name: los_days, dtype: float64
num visits: 62635
num zero LOS: 13128
pct zero LOS: 0.20959527420771135
max LOS: 170


## Sanity Check

This checks the fields used by `GT-BEHRT_Notebook.ipynb`.

In [11]:
sample = sample_graph
assert sample is not None, 'No sample graph was retained. Run the save cell first.'
print(sample)
print('x:', sample.x.shape, sample.x.dtype, 'max id:', int(sample.x.max()))
print('edge_index:', sample.edge_index.shape, sample.edge_index.dtype)
print('edge_attr:', sample.edge_attr.shape, sample.edge_attr.dtype, 'unique:', sorted(sample.edge_attr.unique().tolist()))

required_fields = ['x', 'edge_index', 'edge_attr', 'age', 'time', 'delta', 'adm_type', 'posi_ids', 'mask_v', 'los', 'label', 'mask']

# posi_ids is added by GDSet in the training notebook, not stored here.
stored_fields = [field for field in required_fields if field != 'posi_ids']
missing = [field for field in stored_fields if not hasattr(sample, field)]
assert not missing, missing
assert sample_patient_len == MAX_SEQ_LEN
assert int(sample.x.min()) >= 0
assert int(sample.x.max()) < vocab_size
assert int(sample.adm_type.min()) >= 0
assert int(sample.adm_type.max()) <= max(adm_type_vocab.values(), default=0)
print('sanity check passed')


Data(x=[17], edge_index=[2, 272], edge_attr=[272], age=[1], time=[1], delta=[1], adm_type=[1], los=[1], mask_v=[1], label=[1], mask=[1])
x: torch.Size([17]) torch.int64 max id: 2271
edge_index: torch.Size([2, 272]) torch.int64
edge_attr: torch.Size([272]) torch.int64 unique: [0, 1, 3, 6]
sanity check passed
